In [1]:
# Core Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Model Selection and Pipelines
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Preprocessing & Feature Engineering
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures, FunctionTransformer
from sklearn.impute import SimpleImputer

# Algorithms (Built-in to sklearn)
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

# Evaluation Metrics (Regression focused)
from sklearn.metrics import (
    mean_absolute_error,    # MAE
    mean_squared_error,     # MSE
    root_mean_squared_error, # RMSE (Note: in newer sklearn versions)
    r2_score                # R-squared
)

# Utils
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_theme(style="whitegrid")
%matplotlib inline

In [2]:
df = pd.read_csv('../data/vehicles.csv')
df.sample(5)

,id,url,region,region_url,price,year,manufacturer,model,condition,cylinders,...,size,type,paint_color,image_url,description,county,state,lat,long,posting_date
408407,7316758959,https://seattle.craigslist.org/tac/ctd/d/puyal...,seattle-tacoma,https://seattle.craigslist.org,30999,2019.0,honda,crv ex awd gas suv auto,NaN,NaN,...,NaN,SUV,white,https://images.craigslist.org/00000_iAZ3Oc4KMN...,2019 Honda CRV EX AWD **Clean Carfax One Owner...,NaN,wa,47.205599,-122.291109,2021-05-04T09:06:07-0700
399506,7313048156,https://winchester.craigslist.org/ctd/d/manass...,winchester,https://winchester.craigslist.org,19495,2014.0,mercedes-benz,ml-class,NaN,NaN,...,NaN,SUV,white,https://images.craigslist.org/00808_jfuTvdfAn2...,2014 MERCEDES-BENZ M-CLASS ML 350 Offered ...,NaN,va,38.762669,-77.461754,2021-04-26T17:33:52-0400
211824,7310028398,https://duluth.craigslist.org/ctd/d/minneapoli...,duluth / superior,https://duluth.craigslist.org,27985,2018.0,chevrolet,silverado 1500 lt z71,excellent,8 cylinders,...,full-size,pickup,white,https://images.craigslist.org/00909_4EGF1lhBCT...,This is the iconic 2016 Chevrolet Silverado 15...,NaN,mn,45.064500,-93.341300,2021-04-20T15:51:59-0500
13395,7304182464,https://prescott.craigslist.org/cto/d/cottonwo...,prescott,https://prescott.craigslist.org,8550,1977.0,chevrolet,blazer,excellent,8 cylinders,...,full-size,SUV,white,https://images.craigslist.org/00I0I_8skSslKRWu...,"1977 K5 4x4 $11,770 Fresh 350CI four bolt ma...",NaN,az,34.575122,-112.117152,2021-04-09T08:48:53-0700
199414,7310040021,https://flint.craigslist.org/cto/d/grand-blanc...,flint,https://flint.craigslist.org,19500,2011.0,cadillac,escalade,excellent,8 cylinders,...,NaN,SUV,white,https://images.craigslist.org/00S0S_aXXNYgRzgP...,Super clean-inside & out! 2nd owner! Oversized...,NaN,mi,42.928200,-83.626400,2021-04-20T17:13:13-0400


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426880 entries, 0 to 426879
Data columns (total 26 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            426880 non-null  int64  
 1   url           426880 non-null  object 
 2   region        426880 non-null  object 
 3   region_url    426880 non-null  object 
 4   price         426880 non-null  int64  
 5   year          425675 non-null  float64
 6   manufacturer  409234 non-null  object 
 7   model         421603 non-null  object 
 8   condition     252776 non-null  object 
 9   cylinders     249202 non-null  object 
 10  fuel          423867 non-null  object 
 11  odometer      422480 non-null  float64
 12  title_status  418638 non-null  object 
 13  transmission  424324 non-null  object 
 14  VIN           265838 non-null  object 
 15  drive         296313 non-null  object 
 16  size          120519 non-null  object 
 17  type          334022 non-null  object 
 18  pain

In [4]:
# id column is not useful and it is unique for each row
# url, region_url, image_url, description are not useful, description is unique long text for each row
#VIN is unique for each car, and it has a lot of missing values, so it is not useful 
# county has 0 non-null values, so it is not useful
# dropping these immidiately
initial_drop_cols = ['id','url', 'region_url', 'image_url', 'description', 'county', 'VIN']
df.drop(columns=initial_drop_cols, inplace=True)

In [5]:
# copying the dataframe to have a backup of the original data
df_copy = df.copy()

In [6]:
df['region'].value_counts()

region
columbus                   3608
jacksonville               3562
spokane / coeur d'alene    2988
eugene                     2985
fresno / madera            2983
                           ... 
meridian                     28
southwest MS                 14
kansas city                  11
fort smith, AR                9
west virginia (old)           8
Name: count, Length: 404, dtype: int64

In [7]:
# region is categorical and has 404 unique values, too much for one hot encoding, so dropping
df.drop(columns=['region'], inplace=True)

In [8]:
print(df['state'].nunique())
df['state'].value_counts()

51


state
ca    50614
fl    28511
tx    22945
ny    19386
oh    17696
or    17104
mi    16900
nc    15277
wa    13861
pa    13753
wi    11398
co    11088
tn    11066
va    10732
il    10387
nj     9742
id     8961
az     8679
ia     8632
ma     8174
mn     7716
ga     7003
ok     6792
sc     6327
mt     6294
ks     6209
in     5704
ct     5188
al     4955
md     4778
nm     4425
mo     4293
ky     4149
ar     4038
ak     3474
la     3196
nv     3194
nh     2981
dc     2970
me     2966
hi     2964
vt     2513
ri     2320
sd     1302
ut     1150
wv     1052
ne     1036
ms     1016
de      949
wy      610
nd      410
Name: count, dtype: int64

In [ ]:
# state has 51 unique values, but we leave it for now, decide later

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426880 entries, 0 to 426879
Data columns (total 18 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   price         426880 non-null  int64  
 1   year          425675 non-null  float64
 2   manufacturer  409234 non-null  object 
 3   model         421603 non-null  object 
 4   condition     252776 non-null  object 
 5   cylinders     249202 non-null  object 
 6   fuel          423867 non-null  object 
 7   odometer      422480 non-null  float64
 8   title_status  418638 non-null  object 
 9   transmission  424324 non-null  object 
 10  drive         296313 non-null  object 
 11  size          120519 non-null  object 
 12  type          334022 non-null  object 
 13  paint_color   296677 non-null  object 
 14  state         426880 non-null  object 
 15  lat           420331 non-null  float64
 16  long          420331 non-null  float64
 17  posting_date  426812 non-null  object 
dtypes: f

In [11]:
df.sample(5)

,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,size,type,paint_color,state,lat,long,posting_date
112495,8500,1987.0,chevrolet,corvette,excellent,8 cylinders,gas,80000.0,clean,automatic,NaN,NaN,NaN,white,fl,37.132840,-95.785580,2021-05-01T14:23:24-0400
43719,13995,2018.0,toyota,yaris ia,NaN,4 cylinders,gas,52465.0,clean,automatic,fwd,sub-compact,sedan,grey,ca,33.679591,-117.908531,2021-04-27T12:05:02-0700
77178,23990,2017.0,jaguar,xe 35t premium sedan 4d,good,6 cylinders,gas,25787.0,clean,other,rwd,NaN,sedan,green,co,39.760000,-104.870000,2021-05-02T13:51:18-0600
24811,10500,2011.0,hyundai,genesis coupe 2.0t,NaN,4 cylinders,gas,108292.0,clean,automatic,rwd,mid-size,coupe,black,ca,38.655305,-120.953858,2021-04-11T11:16:07-0700
141434,7995,2009.0,chevrolet,traverse ls,NaN,6 cylinders,gas,118350.0,clean,automatic,fwd,NaN,SUV,blue,il,41.572305,-88.113993,2021-05-01T12:52:39-0500


In [12]:
# lats and longs: bigger numbers dont necessarily mean more expensive cars, so we can drop them for now, decide later if we want to use them or not
# also have about 8000 missing values
df.drop(columns=['lat', 'long'], inplace=True)

In [16]:
# making the number of cylinders an integer by removing the 'cylinders' string and converting to numeric
df['cylinders'] = df['cylinders'].str.replace('cylinders', '').str.strip()
df['cylinders'] = pd.to_numeric(df['cylinders'], errors='coerce')


In [17]:
df.describe()

,price,year,cylinders,odometer
count,4.268800e+05,425675.000000,247904.000000,4.224800e+05
mean,7.519903e+04,2011.235191,5.968685,9.804333e+04
std,1.218228e+07,9.452120,1.602962,2.138815e+05
min,0.000000e+00,1900.000000,3.000000,0.000000e+00
25%,5.900000e+03,2008.000000,4.000000,3.770400e+04
50%,1.395000e+04,2013.000000,6.000000,8.554800e+04
75%,2.648575e+04,2017.000000,8.000000,1.335425e+05
max,3.736929e+09,2022.000000,12.000000,1.000000e+07


In [ ]:
# loaded the data, 426880 rows
# condition will be filled with 'not_reported' because it has almost 45  percent missing values, and it is a categorical variable.
# cylinders will be filled with the most frequent value for that particular car brand/model. so group by model and take mode
# will use functiontransformer in my pipeline for cylinder and age to fill the missing values.
# will probably reduce the number of states by grouping them into regions, because there are 51 unique values and it is a categorical variable., about 10 regions before one hot encoding.
# will drop the original region column cuz too many unique values and it is a categorical variable.
# will calculate the age of the car and drop the year posted column which i will get from the date posted column.
# will input the mileage with the median/mean value based on model/brand and the age
# might add poly features for age and mileage
